# Eve demo: a calculator

Define tools as plain Python functions with `@tool` and register them once with `add_tools`, then
ask Eve in plain English. Eve picks the tool, writes its arguments, and `run()` calls the function.

Arithmetic is one of the domains held out of Eve's training, so the calculator shows how it does
on **new tools**: expect lower confidences, and misses when a request shares no word with
the tool's name ("plus", "times").

In [1]:
from pathlib import Path

from eve import Eve, tool

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
eve = Eve.load(ROOT / "artifacts/runs/joint-base/model.pt")
eve

Eve('/Users/deniskrylov/Developer/eve-lm/artifacts/runs/joint-base/model.pt')

## Calculator

In [2]:
@tool
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b


@tool
def subtract(a: float, b: float) -> float:
    """Subtract the second number from the first."""
    return a - b


@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b


@tool
def divide(a: float, b: float) -> float:
    """Divide the first number by the second."""
    return a / b


eve.add_tools(add, subtract, multiply, divide)
eve.tools

ToolBox(4 tools)

Ask Eve: it shows the intent, how sure it is, and the call it would make.

In [3]:
r = eve.respond("multiply 6 by 4")
r

<Response call 64%: multiply(a=6.0, b=4.0)>

`run()` makes the call. It refuses (with `EveError`) unless the intent is `call` and every argument is there.

In [4]:
r.run()

[24.0]

More requests:

In [5]:
for text in [
    "add 7 and 5",
    "what is 20 plus 10",
    "subtract 3 from 10",
    "take 5 away from 12",
    "multiply 6 by 4",
    "what's 9 times 8",
    "divide 12 by 3",
    "split 100 between 4 people",
    "what's the weather like",
]:
    print(f"{text!r:30} {eve.respond(text)}")

'add 7 and 5'                  call 39%: add(a=7.0, b=5.0)
'what is 20 plus 10'           unsupported 54%
'subtract 3 from 10'           call 59%: subtract(a=3.0, b=10.0)
'take 5 away from 12'          unsupported 56%
'multiply 6 by 4'              call 64%: multiply(a=6.0, b=4.0)
"what's 9 times 8"             unsupported 53%
'divide 12 by 3'               call 60%: divide(a=12.0, b=3.0)
'split 100 between 4 people'   unsupported 54%
"what's the weather like"      unsupported 54%
